In [3]:
import itertools
import warnings
import json
import matplotlib.pyplot as plt
import pymatgen
from pymatgen.core import Composition
from pymatgen.ext.matproj import MPRester
import smact

# Assuming the SMACT-related code is in a separate file named smact_utils.py
# which includes:
# - smact_filter
# - Oxidation_state_probability_finder
# - smact_validity
# etc.
from smact import Element, element_dictionary
from smact.screening import smact_filter, smact_validity
from smact.oxidation_states import Oxidation_state_probability_finder

###############################################################################
# USER INPUTS
###############################################################################

CHEMICAL_SYSTEM = ["Cu", "Ti", "O"]
MAX_STOICH = 8
MP_API_KEY = "w1FbQcBbkQZJHyhMJGxssIEttJ8JSGu8"  # Replace with your key env version for api safety
MAX_ENERGY_ABOVE_HULL = 0.15  # eV/atom threshold for metastability


In [7]:
# STEP 1: DEFINE CHEMICAL SPACE
elements = [Element(el) for el in CHEMICAL_SYSTEM]

# Generate all possible compositions up to MAX_STOICH
all_compositions = [
    Composition({el.symbol: amt for el, amt in zip(elements, amt_list)}).reduced_composition
    for amt_list in itertools.product(range(MAX_STOICH + 1), repeat=len(elements))
    if max(amt_list) > 0
]

# Remove duplicates after reduction
all_compositions = list(set(all_compositions))

# STEP 2: APPLY CHEMICAL FILTERS (SMACT)
valid_compositions = [comp for comp in all_compositions if smact_validity(comp)]
unique_valid_compositions = list(set(valid_compositions))

print(
    f"Number of unique valid compositions: {len(unique_valid_compositions)} out of {len(all_compositions)}"
)

# Print first 5 valid compositions as examples
print("\nExample valid compositions:")
for comp in unique_valid_compositions[:5]:
    print(f"  {comp}")



Number of unique valid compositions: 147 out of 571

Example valid compositions:
  Cu8 Ti2 O7
  Cu7 Ti1 O5
  Cu7 Ti1 O8
  Cu2 Ti1 O6
  Cu2 Ti1 O7


In [4]:

# STEP 3: OXIDATION STATE FILTERS

# replace this with a loop through ICSD oxidation states and then basically narrow down the search space to 
# the ones that are most likely to match known compounds


NameError: name 'elements' is not defined

In [ ]:

==== 
import plotly.graph_objects as go
# Extract fractional compositions
e1 = np.array([comp[elements[0]] for comp in unique_valid_compositions])
e2 = np.array([comp[elements[1]] for comp in unique_valid_compositions])
e3 = np.array([comp[elements[2]] for comp in unique_valid_compositions])

# Normalize to get fractions
total = e1 + e2 + e3
e1 = e1 / total
e2 = e2 / total
e3 = e3 / total

# plot with line from origin_fig
fig = go.Figure()
# mp entries
fig.add_trace(
    go.Scatterternary(
        a=e1,
        b=e2,
        c=e3,
        mode="markers",
        marker=dict(
            size=6,
            color="green",
            symbol="circle",
            opacity=0.7,
            # line=dict(width=0)
            # line=dict(width=1, color="black"),
        ),
        name="SMACT Valid",
        cliponaxis=True,
    )
)
# layout
axis_style = dict(
    title=dict(font=dict(size=10)),  # Font size for the axis titles
    linewidth=1,  # Line width for the axis lines
    linecolor="black",  # Color of the axis lines
    gridcolor="rgba(128, 128, 128, 0.2)",  # Color of the grid lines with transparency
    showticklabels=True,
    tickvals=[0.2, 0.4, 0.6, 0.8],  # Only show these tick positions
)
fig.update_layout(
    font=dict(size=10, family="Arial"),
    width=210,
    height=210,
    ternary=dict(
        bgcolor="rgba(0, 0, 0, 0)",
        aaxis=dict(axis_style, title="Ti"),
        baxis=dict(axis_style, title="Zn"),
        caxis=dict(axis_style, title="O"),
    ),
    margin=dict(l=20, r=20, b=30, t=20),
    showlegend=False,  ##
    legend=dict(
        x=0.65,  # Position the legend on the far left
        y=1.08,  # Center the legend vertically
    ),
    # plot_bgcolor="rgba(0, 0, 0, 0)",
    # paper_bgcolor="rgba(0, 0, 0, 0)",
)
# Show the plot
fig.show()
# save image
# fig.write_image("figures/4_known_system/TiZnO_smact.svg")


In [ ]:

# STEP 4: VISUALIZE ON PHASE DIAGRAM (TERNARY PLOT)


# Only works straightforwardly for ternary systems
if len(CHEMICAL_SYSTEM) == 3:
    # Convert each composition into fractional coordinates for ternary plotting
    # Normalize amounts to sum to 1
    plot_points = []
    for comp in final_allowed_compositions:
        fractions = [comp.get_atomic_fraction(el) for el in CHEMICAL_SYSTEM]
        plot_points.append(tuple(fractions))

    fig, ax = plt.subplots()
    figure, tax = ternary.figure(scale=1.0)
    tax.boundary(linewidth=2.0)
    tax.gridlines(color="black", multiple=0.1, linewidth=0.5)
    tax.set_title("Allowed Compositions in {} System".format("-".join(CHEMICAL_SYSTEM)), fontsize=16)
    # Plot allowed compositions
    tax.scatter(plot_points, marker='o', color='green', label='Chemical Filter')

    tax.left_axis_label(CHEMICAL_SYSTEM[0], fontsize=14)
    tax.right_axis_label(CHEMICAL_SYSTEM[1], fontsize=14)
    tax.bottom_axis_label(CHEMICAL_SYSTEM[2], fontsize=14)

    tax.legend()
    tax.show()
else:
    warnings.warn("Visualization code only provided for ternary systems.")


In [ ]:

# STEP 5: CHECKING STABILITY AGAINST MATERIALS PROJECT DATA

# Using pymatgen's MPRester to get phase diagram and hull information.
# This requires internet connection and an MP API key.
with MPRester(MP_API_KEY) as mpr:
    # Get phase diagram entries for the system
    entries = mpr.get_entries_in_chemsys(CHEMICAL_SYSTEM)
    from pymatgen.analysis.phase_diagram import PhaseDiagram
    pd = PhaseDiagram(entries)

    stable_comps = []
    metastable_comps = []

    for comp in final_allowed_compositions:
        # Get energy above hull for the composition
        decomp, e_above_hull = pd.get_decomp_and_e_above_hull(pd.get_entry_by_composition(comp))
        if e_above_hull is not None:
            if e_above_hull <= 0.0:
                stable_comps.append((comp, e_above_hull))
            elif e_above_hull <= MAX_ENERGY_ABOVE_HULL:
                metastable_comps.append((comp, e_above_hull))
    
    print("Stable compositions:", stable_comps)
    print("Metastable compositions (within 0.15 eV/atom):", metastable_comps)


In [ ]:

###############################################################################
# STEP 6: (OPTIONAL) MACE-MP EVALUATION (PLACEHOLDER)
###############################################################################

# Pseudocode:
# for comp in metastable_comps:
#     # Generate initial structure(s) for comp (external code).
#     # Use MACE-MP or similar ML potential to relax structure:
#     # relaxed_structure, energy = run_mace_mp(initial_structure)
#     # If energy improved and still metastable/stable, keep it.
#     pass

###############################################################################
# STEP 7: UPDATE PHASE DIAGRAM WITH NEW DATA
###############################################################################

# After MACE-MP refinement and re-calculation of energies above hull,
# you could re-plot or re-check stability and visualize again.

print("Workflow complete.")
